# Graph: readable experiments inside a notebook

This notebook introduces a tiny way to make an analysis flow visible, rerun it
with different settings, and keep every step as ordinary Python.

The example is deliberately **descriptive**. It uses one compact recording to
teach the flow; it does not test a team hypothesis, fit a model, rank stimulus
roles, or make a biological claim.

**Start here:** choose **Runtime → Run all**, scroll to **Configure and run**,
change any hollow input socket, and select **Run flow**.

## What the team gets

- **A visible flow:** each box is a function; curved wires show named data dependencies.
- **Repeatable variations:** change a few settings and rerun the same steps.
- **Readable code:** every scientific operation remains a normal Python function.
- **One run record:** outputs, effective settings, execution order, and timings stay together.

There is no draggable desktop canvas, distributed runner, cache system, or hidden
analysis framework. The small notebook surface is intentional.

## One-time Drive setup

Open the shared **Zhong et al. 2025 - Neuromatch Team Workspace** folder in
Google Drive and choose **Organize → Add shortcut → My Drive**. Keep its
existing name. A dataset shortcut named **`Zhong2025_Janelia_v2`** works too.
You need only one shortcut; do not make a copy.

In [ ]:
import importlib
from pathlib import Path
import subprocess
import sys

try:
    import google.colab  # type: ignore[import-not-found]
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    from google.colab import drive, output as colab_output

    drive.mount('/content/drive', force_remount=False)
    MY_DRIVE = Path('/content/drive/MyDrive')
    DATASET_CHOICES = {
        'dataset shortcut': MY_DRIVE / 'Zhong2025_Janelia_v2',
        'workspace shortcut': (
            MY_DRIVE
            / 'Zhong et al. 2025 - Neuromatch Team Workspace'
            / 'Janelia dataset - Zhong et al. 2025 (Figshare v2)'
        ),
    }
    match = next(
        ((label, path) for label, path in DATASET_CHOICES.items() if path.is_dir()),
        None,
    )
    if match is None:
        raise FileNotFoundError(
            'Shared data not found. In Google Drive, add either the team workspace '
            'or dataset folder as a shortcut in My Drive, then rerun this cell.'
        )
    SHORTCUT_KIND, DATASET_ROOT = match
    wheels = sorted((DATASET_ROOT / 'team_tools/packages').glob('zhong2025-*.whl'))
    if len(wheels) != 1:
        raise FileNotFoundError(
            'Expected one zhong2025 helper bundle in team_tools/packages'
        )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-deps",
            "--force-reinstall",
            str(wheels[0]),
        ],
        check=True,
    )
    for module_name in list(sys.modules):
        if (
            module_name == 'graph'
            or module_name == 'zhong2025'
            or module_name.startswith('zhong2025.')
        ):
            del sys.modules[module_name]
    importlib.invalidate_caches()
    colab_output.enable_custom_widget_manager()
    print(f'Shared data: {DATASET_ROOT} ({SHORTCUT_KIND})')

print("Ready. Dataset downloads: 0")

## The four-word mental model

1. **Node** — one Python function and one understandable step.
2. **Input port** — a named function argument such as `demo` or `summary`.
3. **Output port** — a named value declared with `@graph.node(outputs=...)`.
4. **Graph** — nodes run in the order shown; matching port names connect them.

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import graph
from zhong2025 import load_atlas_demo

demo_preview = load_atlas_demo()
{
    "recording": demo_preview["metadata"]["session"],
    "trials": len(demo_preview["trial_id"]),
    "position_bins": len(demo_preview["position_centers_m"]),
    "published_pc_features": demo_preview["population_features"].shape[-1],
    "observed_stimulus_ids": sorted(np.unique(demo_preview["stimulus_id"]).tolist()),
}

## The sample experiment

Pedagogical prompt: **How does the same compact recording look under different
valid descriptive views?**

The flow loads the bundled example, checks its aligned axes and provenance,
selects trials and a corridor region, summarizes one published population
component with running speed as context, and draws the result. The quality
check and trial selection form two visible independent branches that rejoin at the summary.
They run one at a time in numbered order; the branch is logical, not a hidden
parallel computation. Position is already binned into 18 bins in the compact
example. Each trial/bin value is a mean over **moving frames only**, and the
flow never changes that preprocessing.

In [ ]:
#@title Five ordinary Python steps { display-mode: "form" }
@graph.node(outputs="demo")
def load_compact_recording():
    """Load the small, pickle-free recording included with the package."""
    return load_atlas_demo()


@graph.node(outputs="quality")
def check_recording(demo):
    """Verify aligned axes, coverage, and source provenance before analysis."""
    population = np.asarray(demo["population_features"])
    speed = np.asarray(demo["mean_run_speed"])
    frame_counts = np.asarray(demo["frame_counts"])
    expected = (len(demo["trial_id"]), len(demo["position_centers_m"]))
    if population.shape[:2] != speed.shape or speed.shape != frame_counts.shape:
        raise ValueError("population, speed, and frame-count axes do not align")
    if speed.shape != expected:
        raise ValueError(f"expected trial × position shape {expected}, got {speed.shape}")

    metadata = demo["metadata"]
    return {
        "session": metadata["session"],
        "source": metadata["source"],
        "doi": metadata["doi"],
        "source_file_ids": tuple(metadata["source_file_ids"]),
        "shape": population.shape,
        "zero_frame_trial_bins": int(np.count_nonzero(frame_counts <= 0)),
        "finite_population_fraction": float(np.isfinite(population).mean()),
    }


@graph.node(outputs="selection")
def select_trials(demo, stimulus_id=None, position_scope="full"):
    """Choose a stimulus role and corridor region without changing the data."""
    observed = sorted(np.unique(demo["stimulus_id"]).tolist())
    if stimulus_id is None:
        trial_mask = np.ones(len(demo["trial_id"]), dtype=bool)
        stimulus_label = "all observed stimulus roles"
    else:
        if stimulus_id not in observed:
            raise ValueError(f"stimulus_id must be one of {observed} or None")
        trial_mask = demo["stimulus_id"] == stimulus_id
        stimulus_label = f"stimulus role {stimulus_id}"

    texture = np.asarray(demo["texture_bin_mask"], dtype=bool)
    scopes = {
        "full": np.ones_like(texture),
        "texture": texture,
        "gray": ~texture,
    }
    if position_scope not in scopes:
        raise ValueError(f"position_scope must be one of {list(scopes)}")

    return {
        "trial_mask": trial_mask,
        "bin_mask": scopes[position_scope],
        "stimulus_label": stimulus_label,
        "position_scope": position_scope,
        "observed_ids": observed,
    }


@graph.node(outputs="summary")
def summarize_position_profiles(demo, quality, selection, pc_index=0, statistic="mean"):
    """Summarize moving-frame-conditioned component and speed values."""
    population = np.asarray(demo["population_features"])
    if not 0 <= pc_index < population.shape[-1]:
        raise ValueError(f"pc_index must be between 0 and {population.shape[-1] - 1}")
    if statistic not in {"mean", "median"}:
        raise ValueError("statistic must be 'mean' or 'median'")

    trials = selection["trial_mask"]
    bins = selection["bin_mask"]
    neural = population[trials][:, bins, pc_index]
    speed = np.asarray(demo["mean_run_speed"])[trials][:, bins]
    frame_counts = np.asarray(demo["frame_counts"])[trials][:, bins]
    reducer = np.nanmean if statistic == "mean" else np.nanmedian

    return {
        "position_m": np.asarray(demo["position_centers_m"])[bins],
        "neural_center": reducer(neural, axis=0),
        "neural_q25": np.nanquantile(neural, 0.25, axis=0),
        "neural_q75": np.nanquantile(neural, 0.75, axis=0),
        "speed_center": reducer(speed, axis=0),
        "speed_q25": np.nanquantile(speed, 0.25, axis=0),
        "speed_q75": np.nanquantile(speed, 0.75, axis=0),
        "minimum_frames_per_trial_bin": int(frame_counts.min()),
        "zero_frame_trial_bins": int(np.count_nonzero(frame_counts <= 0)),
        "minimum_valid_trials_per_bin": int(np.isfinite(neural).sum(axis=0).min()),
        "finite_neural_fraction": float(np.isfinite(neural).mean()),
        "trial_count": int(trials.sum()),
        "session": quality["session"],
        "doi": quality["doi"],
        "stimulus_label": selection["stimulus_label"],
        "position_scope": selection["position_scope"],
        "pc_index": int(pc_index),
        "statistic": statistic,
    }


@graph.node(outputs="figure")
def plot_position_profiles(summary):
    """Draw a descriptive view; return the figure as another named value."""
    x = summary["position_m"]
    fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True, constrained_layout=True)

    axes[0].plot(x, summary["neural_center"], marker="o", label="center")
    axes[0].fill_between(
        x,
        summary["neural_q25"],
        summary["neural_q75"],
        alpha=0.2,
        label="trial IQR",
    )
    axes[0].set_ylabel(
        f"published PC {summary['pc_index'] + 1} "
        f"(moving-frame means; across-trial {summary['statistic']})"
    )
    axes[0].legend()

    axes[1].plot(x, summary["speed_center"], marker="s", label="center")
    axes[1].fill_between(
        x,
        summary["speed_q25"],
        summary["speed_q75"],
        alpha=0.2,
        label="trial IQR",
    )
    axes[1].set(
        xlabel="corridor position (m)",
        ylabel=(
            "running speed (moving-frame means; "
            f"across-trial {summary['statistic']}; release units)"
        ),
    )
    axes[1].legend()

    fig.suptitle(
        f"{summary['session']} · illustrative moving-frame-conditioned view\n"
        f"{summary['stimulus_label']} · {summary['position_scope']} corridor · "
        f"{summary['trial_count']} trials · {summary['statistic']}"
    )
    return fig

## Build the flow

In [ ]:
#@title Build the flow { display-mode: "form" }
experiment = graph.Graph(
    "Descriptive position profiles",
    load_compact_recording,
    check_recording,
    select_trials,
    summarize_position_profiles,
    plot_position_profiles,
)
experiment.validate()

## Configure and run

This is the notebook's main surface. Curved wires show the actual data path.
**Filled input sockets** receive values from earlier nodes; **Hollow input
sockets** contain settings you can change directly inside their nodes. Choose
settings, select **Run flow**,
and inspect completed steps, safe value previews, and the result together.
The resulting Python record is also available as `run_panel.last_run`.

For a team walkthrough, rotate three lightweight roles: one person changes and
runs the ports, one explains the biological and axis meaning, and one checks the
quality/provenance branch. Everyone should be able to take each role.

In [ ]:
#@title Configure the visible ports { display-mode: "form" }
run_panel = experiment.widget(
    controls={
        "stimulus_id": widgets.Dropdown(
            description="Stimulus role",
            options=[
                ("All roles", None),
                ("Role 0 — familiar B exemplar", 0),
                ("Role 1 — novel B exemplar", 1),
                ("Role 2 — familiar A exemplar", 2),
                ("Role 3 — novel A exemplar", 3),
            ],
            value=None,
        ),
        "position_scope": widgets.Dropdown(
            description="Corridor region",
            options=[
                ("Whole corridor", "full"),
                ("Texture region", "texture"),
                ("Gray region", "gray"),
            ],
            value="full",
        ),
        "pc_index": widgets.Dropdown(
            description="Published component",
            options=[(f"PC {index + 1}", index) for index in range(6)],
            value=0,
        ),
        "statistic": widgets.Dropdown(
            description="Summary",
            options=[("Mean", "mean"), ("Median", "median")],
            value="mean",
        ),
    },
    show="figure",
)
run_panel

## Optional: run the same flow in Python

The defaults include all observed stimulus roles, the full corridor, the first
published component, and a mean summary. This is an orientation run, not a
preferred analysis. The code path and the port UI call the same flow.

In [ ]:
baseline = experiment.run(until="summary")
print("Order:", " → ".join(baseline.order))
print("Settings:", dict(baseline.settings))
print("Trials:", baseline["summary"]["trial_count"])
print("Minimum contributing moving frames in any selected trial/bin:", baseline["summary"]["minimum_frames_per_trial_bin"])
print("Zero-frame selected trial/bins:", baseline["summary"]["zero_frame_trial_bins"])
print("Minimum valid trials per selected position bin:", baseline["summary"]["minimum_valid_trials_per_bin"])
print("Source:", baseline["quality"]["doi"])

## Run explicit variations

`run_many` takes an ordered list. It does not silently build a large parameter
grid. Here every observed stimulus role receives the same descriptive settings;
the table checks coverage and shapes rather than scoring or ranking the roles.

In [ ]:
observed_ids = sorted(np.unique(baseline["demo"]["stimulus_id"]).tolist())
variation_settings = [{"stimulus_id": value} for value in observed_ids]
variation_runs = experiment.run_many(variation_settings, until="summary")

variation_table = [
    {
        "stimulus_id": run.settings["stimulus_id"],
        "trials": run["summary"]["trial_count"],
        "position_bins": len(run["summary"]["position_m"]),
        "zero_frame_bins": run["summary"]["zero_frame_trial_bins"],
        "minimum_valid_trials_per_bin": run["summary"]["minimum_valid_trials_per_bin"],
        "pc_index": run.settings["pc_index"],
        "statistic": run.settings["statistic"],
    }
    for run in variation_runs
]
variation_table

## Inspect a run instead of trusting hidden state

Each run owns its settings, named outputs, terminal branches, order, and timings. A teammate can
inspect intermediate values such as `selection` or `summary` without rerunning
ad hoc notebook cells.

In [ ]:
{
    "settings": dict(baseline.settings),
    "output_ports": list(baseline.outputs),
    "terminal_ports": list(baseline.terminal_ports),
    "order": list(baseline.order),
    "milliseconds_by_node": {
        name: round(seconds * 1000, 2)
        for name, seconds in baseline.timings.items()
    },
}

## Write another sequential flow

The reusable pattern is deliberately small:

```python
@graph.node(outputs="raw_data")
def load():
    return ...

@graph.node(outputs="clean_data")
def clean(raw_data, threshold=0.5):
    return ...

@graph.node(outputs="result")
def summarize(clean_data, method="mean"):
    return ...

my_flow = graph.Graph("My flow", load, clean, summarize)
my_flow.run(threshold=0.7)
```

The output names `raw_data` and `clean_data` match the next function arguments,
so those ports are wired. `threshold` and `method` have defaults, so they become
visible settings.

## Boundaries of this example

- It contains one recording from one mouse. Trials are not independent animals
  and cannot support group-level inference.
- The 48 population features are published session components. The basis is
  label-free but was fit to the session before trial selection, so it is
  transductive and component axes should not be compared across recordings.
- The compact area features are not individual-neuron activity; this notebook
  does not use them.
- Raw lick, cue, reward, neuron-coordinate, full 400-component, and other-session
  data are not included in the compact example.
- Running speed remains in the release's units.
- Neural values, speed values, and contributing-frame counts all describe
  moving frames only; immobility is not represented in these profiles.
- Changing a selector demonstrates a reproducible variation. It is not a
  statistical comparison or a biological conclusion.
- This notebook fits or selects no model, so it needs no train/test split. Any
  later fitting, tuning, decoding, or feature selection flow should make its
  train/test split a visible upstream node and keep test data out of selection.
- Nodes should treat received arrays and dictionaries as read-only. This keeps
  sibling branches deterministic without copying large neural arrays.
- `run_many` reruns every node, including loaders. That is fine for this 2.9 MB
  example; a full-data workflow should load one selected slice once and pass it
  explicitly rather than relying on hidden caching.

Use the data-atlas notebook for the complete experiment and file relationships.
Only turn a stable repeated analysis step into a node; exploratory scratch work
can remain a normal notebook cell.